## 1. Factorize и One-Hot Encoding (OHE)

Оба метода используются для **перевода текстовых (категориальных) данных в числовой формат**, так как большинство алгоритмов машинного обучения не умеют работать со строками.

### Как они работают?
* **`pd.factorize()`**: Заменяет каждое уникальное текстовое значение на порядковый индекс (целое число от $0$ до $N-1$).
* **One-Hot Encoding (OHE)**: Создает для каждого уникального значения отдельный бинарный столбец. В строке ставится `1`, если категория присутствовала, и `0` во всех остальных новых столбцах.

### В чём разница и когда применяются?
* **Факторизация (`factorize`)**: 
  * *Плюсы:* Не увеличивает количество столбцов в таблице (сохраняет её форму).
  * *Минусы:* Создает ложное ощущение математического порядка и иерархии (например, если цвета закодировать как `Красный=0, Синий=1, Зеленый=2`, модель может подумать, что зеленый «больше» красного).
  * *Применение:* Для древовидных моделей (деревья решений, случайный лес) или для категорий с естественным порядком (`низкий=0, средний=1, высокий=2`).
* **One-Hot Encoding (OHE)**:
  * *Плюсы:* Исключает ложный порядок категорий.
  * *Минусы:* Сильно увеличивает размер таблицы (проблема разреженности), если уникальных категорий очень много.
  * *Применение:* Для линейных моделей, логистической регрессии и нейросетей, особенно для категорий без порядка (например, города, марки машин).

### Пример сравнения
Для столбца `Город` со значениями: `["Баку", "Гянджа", "Баку"]`:

* **Результат `pd.factorize()`:**  
  `[0, 1, 0]` (где `0` — Баку, `1` — Гянджа). Получился **один** столбец.

* **Результат One-Hot Encoding:**  
  Получится **два** новых столбца:
  
  | Баку | Гянджа |
  | :---: | :---: |
  | 1 | 0 |
  | 0 | 1 |
  | 1 | 0 |

---

## 2. SimpleImputer

### Для чего используется?
`SimpleImputer` из библиотеки `scikit-learn` предназначен для **заполнения пропущенных значений (`NaN`, `None`)** в данных числовыми или текстовыми заглушками.

### Как работают стратегии?
* **`mean`**: Заменяет пропуски на **среднее арифметическое** по столбцу (только для чисел). Чувствительна к выбросам.
* **`median`**: Заменяет пропуски на **медиану** (центральное число отсортированного ряда). Устойчива к выбросам и аномалиям.
* **`most_frequent`**: Заменяет пропуски на **моду** — самое часто встречающееся значение (идеально для текста/категорий).
* **`constant`**: Заменяет все пропуски на заранее заданное пользователем значение (например, `fill_value=0` или `fill_value="Неизвестно"`).

### Примеры заполнения
* *Зарплаты сотрудников:* `[1000, NaN, 1200, 5000]` $\rightarrow$ Стратегия `median` подставит `1100`, а `mean` выдаст искаженный результат из-за выброса в 5000.
* *Города проживания:* `["Баку", NaN, "Баку", "Сумгаит"]` $\rightarrow$ Стратегия `most_frequent` подставит `"Баку"`, так как он встречается чаще всего.

---

## 3. MinMaxScaler и StandardScaler

### Зачем нужно масштабирование признаков?
Если один признак измеряется тысячами (например, зарплата в AZN), а другой единицами (например, стаж от 1 до 10 лет), модель придаст слишком большой вес первому признаку просто из-за его числового масштаба. Масштабирование приводит все признаки к единому диапазону или распределению.

### Как работает каждый метод (формулы):

1. **MinMaxScaler**: Сжимает все данные в фиксированный диапазон от $0$ до $1$.
   $$\mathbf{x_{scaled}} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

2. **StandardScaler**: Стандартизирует данные так, чтобы среднее значение стало равно $0$, а стандартное отклонение — $1$.
   $$\mathbf{x_{scaled}} = \frac{x - \mu}{\sigma}$$
   *(где $\mu$ — среднее арифметическое, а $\sigma$ — стандартное отклонение).*

### В чём разница между ними?
* **MinMaxScaler** жестко ограничивает данные рамками $[0, 1]$. **Минус:** аномально большие выбросы могут «сплюснуть» все остальные данные в очень узкий диапазон.
* **StandardScaler** не имеет жестких границ, а распределяет данные вокруг нуля. Он лучше справляется с аномалиями и выбросами.

### Почему `fit` нужно выполнять только на обучающих данных?
Метод `.fit()` вычисляет параметры (минимум/максимум или среднее/отклонение). Если вызвать `.fit()` на всём датасете (включая тестовую выборку), произойдет **утечка данных (Data Leakage)** — модель «подглядит» характеристики тестовых данных, и оценки качества на тесте будут нечестно завышенными.  
*Сначала делаем `.fit_transform()` на обучающей (train) выборке, а на тестовой — только `.transform()`.*

---

## 4. Метрики регрессии: MAE, MSE, RMSE, R²

### 1. MAE (Mean Absolute Error — Средняя абсолютная ошибка)
* **Формула:** $\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$
* **Смысл:** Средняя абсолютная разница между реальными ($y$) и предсказанными ($\hat{y}$) значениями. Измеряется в тех же единицах, что и целевая переменная.
* **Лучшее значение:** $0$.

### 2. MSE (Mean Squared Error — Средняя квадратичная ошибка)
* **Формула:** $\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$
* **Смысл:** Ошибки возводятся в квадрат, поэтому крупные промахи наказываются значительно сильнее мелких.
* **Лучшее значение:** $0$.

### 3. RMSE (Root Mean Squared Error — Корень из средней квадратичной ошибки)
* **Формула:** $\text{RMSE} = \sqrt{\text{MSE}} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$
* **Смысл:** Корень из MSE возвращает единицы измерения к исходным (например, AZN), сохраняя жесткое наказание за большие ошибки.
* **Лучшее значение:** $0$.

### 4. R² (Коэффициент детерминации)
* **Формула:** $R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$ *(где $\bar{y}$ — среднее значение целевой переменной).*
* **Смысл:** Показывает долю дисперсии целевой переменной, объясненную моделью (качество подгонки модели относительно простого среднего).
* **Лучшее значение:** $1$ (идеальная модель).

---

### Аналитические вопросы по метрикам:
* **Какие метрики сильнее реагируют на большие ошибки?**  
  **MSE и RMSE**, так как возведение ошибок в квадрат в их формулах существенно увеличивает вес крупных промахов.
* **Может ли $R^2$ быть отрицательным?**  
  **Да, может.** Это происходит тогда, когда модель работает **хуже**, чем простые предсказания среднего арифметического значения целевой переменной для всех объектов.

---

## 5. Cross-Validation (Кросс-валидация)

### Что такое кросс-валидация и зачем она нужна?
Кросс-валидация — это метод оценки качества модели, при котором датасет многократно делится на обучающую и проверочную части. Она необходима, чтобы убедиться, что полученная точность модели не является результатом случайного "успешного" или "неудачного" одного разбиения.

### Как работает K-Fold (на примере `cv=5`)?
1. Исходный датасет делится на **5 равных частей** (фолдов).
2. Процесс повторения происходит **5 раз**:
   * *Шаг 1:* Фулд №1 используется для теста, фолды №2–5 — для обучения.
   * *Шаг 2:* Фулд №2 используется для теста, фолды №1, 3, 4, 5 — для обучения.
   * ...и так далее, пока каждая из 5 частей по очереди не побудет тестовой.
3. Итоговая метрика рассчитывается как среднее арифметическое по всем 5 итерациям.

### Чем кросс-валидация отличается от одного разделения Train/Test?
* **Train/Test Split (однократное деление):** Делит данные всего один раз. Результат сильно зависит от случайного распределения объектов между выборками.
* **Cross-Validation (кросс-валидация):** Тестирует модель на **всех** данных по очереди, давая максимально честную, устойчивую и объективную оценку обобщающей способности алгоритма.